# CS313 Nhóm 5 – E2: chạy DAMR như code công bố (3 dataset × 10 seed × 40 epoch)

**Trước khi chạy:**
1. *Settings → Accelerator*: **GPU T4 x1**. *Settings → Internet*: **On**.
2. *Add Input*: dataset chứa `psm/`, `msl/`, `smd/` (ví dụ `cs313-damr-data`). Notebook tự tìm file theo tên.
3. Kiểm tra `BRANCH` ở ô dưới.
4. Chạy bằng **Save Version → Save & Run All (Commit)**, KHÔNG chạy trong phiên tương tác: chế độ commit chạy nền tới 12 giờ và giữ lại output.

**Khi xong:** tab *Output* của phiên bản → tải `e2_outputs.zip` (CSV, log, checkpoint, điểm `.npy`).
CSV đưa lên GitHub qua PR; checkpoint đưa lên Drive chung, giữ nguyên cấu trúc `checkpoints/E2/...`.

In [ ]:
REPO = "https://github.com/Waterbear566/cs313-damr-reproduce.git"
BRANCH = "pipeline-common"   # đổi thành "main" sau khi merge PR #3
DATASETS = ["PSM", "MSL", "SMD"]
SEEDS = "16 17 18 19 20 21 22 23 24 25"   # 16–18 = seed chung (bảng chính); 16–25 = 10 lần như bài báo
!rm -rf /kaggle/working/repo && git clone -q -b $BRANCH $REPO /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -1

## 1. Môi trường

In [ ]:
# Kaggle đã có torch, numpy, pandas, scikit-learn. Chỉ cài phần còn thiếu, KHÔNG cài lại torch.
!pip install -q thop faiss-cpu lightning timm 2>&1 | tail -2
import platform, subprocess, torch, numpy, sklearn, pandas
print("Python   ", platform.python_version())
print("PyTorch  ", torch.__version__, "| CUDA", torch.version.cuda, "| cuDNN", torch.backends.cudnn.version())
print("numpy    ", numpy.__version__, "| pandas", pandas.__version__, "| sklearn", sklearn.__version__)
print("GPU      ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
assert torch.cuda.is_available(), "Chưa bật GPU: Settings → Accelerator → GPU T4"
!pip freeze > /kaggle/working/environment_kaggle.txt

## 2. Dữ liệu: tìm file trong /kaggle/input và nối vào `DAMR/data/`

In [ ]:
import glob, os
NEED = {"psm": ["train.csv", "test.csv", "test_label.csv"],
        "msl": ["MSL_train.npy", "MSL_test.npy", "MSL_test_label.npy"],
        "smd": ["SMD_train.npy", "SMD_test.npy", "SMD_test_label.npy"]}
for ds, files in NEED.items():
    if ds.upper() not in DATASETS:
        continue
    cands = [d for d in {os.path.dirname(p) for p in glob.glob(f"/kaggle/input/**/{files[0]}", recursive=True)}
             if all(os.path.exists(os.path.join(d, f)) for f in files)]
    if ds == "psm":   # tên train.csv rất phổ biến: ưu tiên thư mục có chữ psm
        cands = sorted(cands, key=lambda d: "psm" not in d.lower())
    assert cands, f"{ds.upper()}: không thấy trong /kaggle/input, kiểm tra Add Input"
    os.makedirs(f"DAMR/data/{ds}", exist_ok=True)
    for f in files:
        dst = f"DAMR/data/{ds}/{f}"
        if not os.path.exists(dst):
            os.symlink(os.path.join(cands[0], f), dst)
    print(f"{ds.upper()}: dùng {cands[0]}")
DS_ARGS = " ".join(DATASETS)
!python scripts/check_data.py --root DAMR/data --datasets $DS_ARGS

## 3. Chạy E2
Mỗi dataset chạy 3 seed bằng `scripts/run_e2.sh` (learning rate theo tác giả, `--log_extra=True`).
Một dataset lỗi thì các dataset sau vẫn chạy; xem log ở `logs/E2_<dataset>_seed<k>.log`.

In [ ]:
import time
os.environ["SEEDS"] = SEEDS
timing = {}
for ds in DATASETS:
    t0 = time.time()
    !bash scripts/run_e2.sh {ds} 0
    timing[ds] = (time.time() - t0) / 60
    print(f"===== {ds} xong sau {timing[ds]:.1f} phút =====")
print(timing)

## 4. Tổng hợp: mean ± std, `best_test` (như code gốc) và `last` (epoch cuối)

In [ ]:
# Bảng chính: 3 seed chung với baseline. Bảng phụ: đủ 10 seed như bài báo.
!python scripts/summarize.py results/E2_DAMR.csv --seeds 16 17 18 --out results/E2_summary.csv
!python scripts/summarize.py results/E2_DAMR.csv --out results/E2_summary_10seeds.csv

## 5. Đóng gói để tải về (tab Output → `e2_outputs.zip`)

In [ ]:
import shutil, zipfile
shutil.copy("/kaggle/working/environment_kaggle.txt", "results/")
with zipfile.ZipFile("/kaggle/working/e2_outputs.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for top in ["results", "logs", "checkpoints/E2"]:
        for root, _, files in os.walk(top):
            for f in files:
                z.write(os.path.join(root, f))   # đường dẫn tương đối gốc repo, dấu /
print(f"e2_outputs.zip: {os.path.getsize('/kaggle/working/e2_outputs.zip') / 1e6:.1f} MB")
!du -sh checkpoints/E2/*